
# Roxy notebook example: k-mers with reduced amino acid alphabets

This notebook is a **reference implementation example** for **reduced-alphabet k-mer descriptors** in Roxy.

Reduced alphabets group amino acids into physicochemical classes to:

- reduce dimensionality
- improve generalization
- retain biological meaning

## Covered outputs

- sequence cleaning
- mapping sequences to reduced alphabets
- definition of multiple reduced alphabets
- k-mer extraction in reduced space
- counts and frequencies
- comparison with full alphabet
- class-style implementation

This is a key step toward making k-mers usable in real ML workflows.


In [1]:

from collections import Counter
from itertools import product

import numpy as np
import pandas as pd


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": ["r1", "r2", "r3", "r4"],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
        ],
    }
)

df_demo


,sequence_id,sequence
0,r1,MKWVTFISLLFLFSSAYSRGVFRR
1,r2,GGGGGGGGGGGGGGG
2,r3,KRRKRRKRRKRRDDDDEE
3,r4,ACDEFGHIKLMNPQRSTVWY


## Standard amino acids

In [3]:

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")


## Reduced alphabets

In [4]:

# Example 1: simple physicochemical grouping
REDUCED_ALPHA_5 = {
    **{aa: "H" for aa in "AVLIMFWY"},   # hydrophobic
    **{aa: "P" for aa in "STNQ"},       # polar
    **{aa: "C" for aa in "KRH"},        # positive
    **{aa: "N" for aa in "DE"},         # negative
    **{aa: "S" for aa in "CGP"},        # special
}

# Example 2: very compact alphabet (3 groups)
REDUCED_ALPHA_3 = {
    **{aa: "H" for aa in "AVLIMFWY"},
    **{aa: "P" for aa in "STNQKRHDE"},
    **{aa: "S" for aa in "CGP"},
}


## Helper functions

In [5]:

def clean_sequence(seq):
    if pd.isna(seq):
        return ""
    seq = str(seq).upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA])


def map_to_reduced(seq, mapping):
    return "".join([mapping.get(aa, "") for aa in seq])


def generate_kmers(alphabet, k):
    return ["".join(p) for p in product(alphabet, repeat=k)]


def extract_kmers(seq, k):
    if len(seq) < k:
        return []
    return [seq[i:i+k] for i in range(len(seq)-k+1)]


def kmer_freq(seq, k, alphabet):
    kmers = extract_kmers(seq, k)
    total = len(kmers)
    all_kmers = generate_kmers(alphabet, k)

    if total == 0:
        return {f"rk{k}_freq_{km}": np.nan for km in all_kmers}

    counts = Counter(kmers)
    return {f"rk{k}_freq_{km}": counts.get(km, 0)/total for km in all_kmers}


## Core descriptor function

In [6]:

def reduced_kmer_descriptors(seq, mapping, k=2):
    seq = clean_sequence(seq)
    reduced_seq = map_to_reduced(seq, mapping)
    alphabet = sorted(set(mapping.values()))

    freqs = kmer_freq(reduced_seq, k, alphabet)

    out = {
        f"rk{k}_length": len(reduced_seq),
        f"rk{k}_alphabet_size": len(alphabet),
    }

    out.update(freqs)
    return out


## Apply example (k=2, alpha=5)

In [7]:

df_rk = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(lambda x: reduced_kmer_descriptors(x, REDUCED_ALPHA_5, k=2)).apply(pd.Series),
    ],
    axis=1
)

df_rk.head()


,sequence_id,sequence,rk2_length,rk2_alphabet_size,rk2_freq_CC,rk2_freq_CH,rk2_freq_CN,rk2_freq_CP,rk2_freq_CS,rk2_freq_HC,...,rk2_freq_PC,rk2_freq_PH,rk2_freq_PN,rk2_freq_PP,rk2_freq_PS,rk2_freq_SC,rk2_freq_SH,rk2_freq_SN,rk2_freq_SP,rk2_freq_SS
0,r1,MKWVTFISLLFLFSSAYSRGVFRR,24.0,5.0,0.043478,0.043478,0.000000,0.000000,0.043478,0.086957,...,0.043478,0.130435,0.0,0.043478,0.000000,0.000000,0.043478,0.000000,0.000000,0.0
1,r2,GGGGGGGGGGGGGGG,15.0,5.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
2,r3,KRRKRRKRRKRRDDDDEE,18.0,5.0,0.647059,0.000000,0.058824,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
3,r4,ACDEFGHIKLMNPQRSTVWY,20.0,5.0,0.000000,0.105263,0.000000,0.052632,0.000000,0.052632,...,0.052632,0.052632,0.0,0.052632,0.052632,0.052632,0.000000,0.052632,0.052632,0.0


## Sparse inspection

In [8]:

cols = [c for c in df_rk.columns if c.startswith("rk2_freq_")]

row = df_rk.loc[0, cols]
row[row > 0].sort_values(ascending=False).head(10)


rk2_freq_HH    0.347826
rk2_freq_HP    0.173913
rk2_freq_PH    0.130435
rk2_freq_HC    0.086957
rk2_freq_CH    0.043478
rk2_freq_CC    0.043478
rk2_freq_CS    0.043478
rk2_freq_PC    0.043478
rk2_freq_PP    0.043478
rk2_freq_SH    0.043478
Name: 0, dtype: float64

## Compare dimensionality

In [9]:

print("Full alphabet k=2:", 20**2)
print("Reduced alpha=5 k=2:", 5**2)
print("Reduced alpha=3 k=2:", 3**2)


Full alphabet k=2: 400
Reduced alpha=5 k=2: 25
Reduced alpha=3 k=2: 9


## Class-style implementation

In [10]:

class ReducedKmerDescriptors:

    def __init__(self, mapping, k=2):
        self.mapping = mapping
        self.k = k
        self.alphabet = sorted(set(mapping.values()))

    def transform_sequence(self, seq):
        return reduced_kmer_descriptors(seq, self.mapping, self.k)

    def transform(self, sequences):
        return pd.DataFrame([self.transform_sequence(s) for s in sequences])


rk_transformer = ReducedKmerDescriptors(REDUCED_ALPHA_5, k=2)
rk_matrix = rk_transformer.transform(df_demo["sequence"])
rk_matrix.head()


,rk2_length,rk2_alphabet_size,rk2_freq_CC,rk2_freq_CH,rk2_freq_CN,rk2_freq_CP,rk2_freq_CS,rk2_freq_HC,rk2_freq_HH,rk2_freq_HN,...,rk2_freq_PC,rk2_freq_PH,rk2_freq_PN,rk2_freq_PP,rk2_freq_PS,rk2_freq_SC,rk2_freq_SH,rk2_freq_SN,rk2_freq_SP,rk2_freq_SS
0,24,5,0.043478,0.043478,0.000000,0.000000,0.043478,0.086957,0.347826,0.0,...,0.043478,0.130435,0.0,0.043478,0.000000,0.000000,0.043478,0.000000,0.000000,0.0
1,15,5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
2,18,5,0.647059,0.000000,0.058824,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
3,20,5,0.000000,0.105263,0.000000,0.052632,0.000000,0.052632,0.157895,0.0,...,0.052632,0.052632,0.0,0.052632,0.052632,0.052632,0.000000,0.052632,0.052632,0.0


In [ ]:

# df_rk.to_csv("demo_reduced_kmer.csv", index=False)
